Import

In [2]:
import pandas as pd 
import numpy as np
import os
import cbbd
import json
from datetime import datetime, timedelta
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
import warnings 
warnings.filterwarnings("ignore")




In [3]:
with open("keys.json", "r") as f:
    keys = json.load(f)
    API_KEY = keys.get("API_KEY")


In [4]:
seasons = range(2020, 2027)
bubble_percentile = 0.15
all_results = []

configuration = cbbd.Configuration(access_token=API_KEY)

def win_prob_vs_bubble(opponent_elo: float, bubble_elo: float) -> float:
    diff = opponent_elo - bubble_elo
    prob = 1.0 / (1.0 + 10 ** (diff / 400))
    return float(np.clip(prob, 0.05, 0.95))

for season in seasons:
    print(f"\nFetching season {season}...")
    year1 = season - 1
    year2 = season
    feb_end = datetime(year2, 3, 1) - timedelta(days=1)

    with cbbd.ApiClient(configuration) as api_client:
        games_raw_1 = cbbd.GamesApi(api_client).get_games(
            start_date_range=datetime(year1, 11, 1),
            end_date_range=datetime(year1, 12, 31),
            status=cbbd.GameStatus.FINAL
        )
        games_raw_2 = cbbd.GamesApi(api_client).get_games(
            start_date_range=datetime(year2, 1, 1),
            end_date_range=feb_end,
            status=cbbd.GameStatus.FINAL
        )
        games_raw_3 = cbbd.GamesApi(api_client).get_games(
            start_date_range=datetime(year2, 3, 1),
            end_date_range=datetime(year2, 3, 14),
            status=cbbd.GameStatus.FINAL
        )
        elo_raw = cbbd.RatingsApi(api_client).get_elo(season=season)

    games_raw = games_raw_1 + games_raw_2 + games_raw_3

    games_list = []
    for game in games_raw:
        games_list.append({
            'homeTeam': game.home_team,
            'awayTeam': game.away_team,
            'homeScore': game.home_points,
            'awayScore': game.away_points,
        })

    games = pd.DataFrame(games_list).drop_duplicates().reset_index(drop=True)
    games["homeScore"] = pd.to_numeric(games["homeScore"], errors="coerce")
    games["awayScore"] = pd.to_numeric(games["awayScore"], errors="coerce")

    elo = pd.DataFrame([rating.to_dict() for rating in elo_raw])
    elo_sorted = elo.sort_values("elo", ascending=False).reset_index(drop=True)
    bubble_index = min(int(len(elo_sorted) * bubble_percentile), len(elo_sorted) - 1)
    bubble_elo = elo_sorted.iloc[bubble_index]["elo"]

    d1_teams = set(elo_sorted["team"])
    games = games[
        (games["homeTeam"].isin(d1_teams)) &
        (games["awayTeam"].isin(d1_teams))
    ].reset_index(drop=True)

    print(f"Season : {season}")
    print(f"Teams  : {len(elo_sorted)}")
    print(f"Games  : {len(games)}")
    print(f"Bubble ELO threshold (top {int(bubble_percentile*100)}%): {bubble_elo:.1f}")

    elo_dict = dict(zip(elo_sorted["team"], elo_sorted["elo"]))
    rank_lookup = {row["team"]: i + 1 for i, row in elo_sorted.iterrows()}

    all_teams = set(games["homeTeam"]) | set(games["awayTeam"])
    results = []

    for team in all_teams:
        team_games = games[
            (games["homeTeam"] == team) | (games["awayTeam"] == team)
        ]

        actual_wins = 0
        expected_wins = 0.0
        games_played = 0

        for _, game in team_games.iterrows():
            if game["homeTeam"] == team:
                opponent  = game["awayTeam"]
                my_score  = game["homeScore"]
                opp_score = game["awayScore"]
            else:
                opponent  = game["homeTeam"]
                my_score  = game["awayScore"]
                opp_score = game["homeScore"]

            if pd.isna(my_score) or pd.isna(opp_score):
                continue

            if my_score > opp_score:
                actual_wins += 1

            opp_elo = elo_dict.get(opponent, 1500)
            expected_wins += win_prob_vs_bubble(opp_elo, bubble_elo)
            games_played += 1

        if games_played == 0:
            continue

        results.append({
            "Season":        season,
            "Team":          team,
            "Wab":           round(actual_wins - expected_wins, 3),
            "Actual_wins":   actual_wins,
            "Expected_wins": round(expected_wins, 3),
            "Games_played":  games_played,
            "Elo":           round(elo_dict.get(team, 1500), 1),
            "Elo_rank":      rank_lookup.get(team),
            "Bubble_elo":    round(bubble_elo, 1)
        })

    all_results.extend(results)
    print(f"Season {season} done — {len(results)} teams calculated")

# Final combined DataFrame — outside the season loop
Wab_data = (
    pd.DataFrame(all_results)
    .sort_values(["Season", "Wab"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\nTop 25 per season:\n")
for season in seasons:
    print(f"\n--- {season} ---")
    season_df = Wab_data[Wab_data["Season"] == season].copy()
    season_df.insert(0, "Rank", range(1, len(season_df) + 1))
    print(season_df.head(25).to_string(index=False))


Fetching season 2020...
Season : 2020
Teams  : 353
Games  : 5198
Bubble ELO threshold (top 15%): 1942.0
Season 2020 done — 353 teams calculated

Fetching season 2021...
Season : 2021
Teams  : 347
Games  : 3776
Bubble ELO threshold (top 15%): 1890.0
Season 2021 done — 347 teams calculated

Fetching season 2022...
Season : 2022
Teams  : 358
Games  : 5314
Bubble ELO threshold (top 15%): 1917.0
Season 2022 done — 358 teams calculated

Fetching season 2023...
Season : 2023
Teams  : 363
Games  : 5473
Bubble ELO threshold (top 15%): 1956.0
Season 2023 done — 363 teams calculated

Fetching season 2024...
Season : 2024
Teams  : 362
Games  : 5415
Bubble ELO threshold (top 15%): 1968.0
Season 2024 done — 362 teams calculated

Fetching season 2025...
Season : 2025
Teams  : 364
Games  : 5485
Bubble ELO threshold (top 15%): 1998.0
Season 2025 done — 364 teams calculated

Fetching season 2026...
Season : 2026
Teams  : 365
Games  : 5420
Bubble ELO threshold (top 15%): 1980.0
Season 2026 done — 365 te

In [11]:
with cbbd.ApiClient(configuration) as api_client:
    srs_sample     = cbbd.RatingsApi(api_client).get_srs(season=2024)
    recruit_sample = cbbd.RecruitingApi(api_client).get_team_recruiting_rankings(year=2024)

print(srs_sample[0].to_dict())
print(recruit_sample[0].to_dict())

{'season': 2024, 'teamId': 314, 'team': 'UConn', 'conference': 'Big East', 'rating': 18.2}
{'teamId': 72, 'team': 'Duke', 'conference': 'ACC', 'year': 2024, 'ranking': 1, 'rating': 70.93}


In [12]:
seasons = range(2020, 2027)
srs_results = []
recruiting_results = []
portal_results = []

with cbbd.ApiClient(configuration) as api_client:
    for season in seasons:
        print(f"Fetching SRS and recruiting for {season}...")
        
        # SRS
        try:
            srs_raw = cbbd.RatingsApi(api_client).get_srs(season=season)
            for s in srs_raw:
                srs_results.append({
                    'Season': s.season,
                    'Team':   s.team,
                    'SRS':    s.rating,
                })
        except Exception as e:
            print(f"  SRS error {season}: {e}")

        # Recruiting
        try:
            rec_raw = cbbd.RecruitingApi(api_client).get_team_recruiting_rankings(year=season)
            for r in rec_raw:
                recruiting_results.append({
                    'Season':              r.year,
                    'Team':                r.team,
                    'Recruiting_Ranking':  r.ranking,
                    'Recruiting_Rating':   r.rating,
                })
        except Exception as e:
            print(f"  Recruiting error {season}: {e}")
        
        # Portal
        try:
            raw = cbbd.RecruitingApi(api_client).get_portal_transfers_with_http_info(
                year=season, _preload_content=False
            )
            import json
            # Access the raw data attribute directly
            transfers = json.loads(raw.raw_data)
            for p in transfers:
                portal_results.append({
                    'Season':      p.get('year'),
                    'origin':      p.get('origin', {}).get('name') if p.get('origin') else None,
                    'destination': p.get('destination', {}).get('name') if p.get('destination') else None,
                    'stars':       p.get('stars'),
                    'rating':      p.get('rating'),
                })
            print(f"  {season}: {len(transfers)} transfers")
        except Exception as e:
            print(f"  Portal error {season}: {e}")


srs_df        = pd.DataFrame(srs_results)
recruiting_df = pd.DataFrame(recruiting_results)
portal_df = pd.DataFrame(portal_results)

print(f"\nSRS rows:        {len(srs_df)}")
print(f"Recruiting rows: {len(recruiting_df)}")
print(f"WAB rows:        {len(Wab_data)}")
print(f"\nPortal rows: {len(portal_df)}")

Fetching SRS and recruiting for 2020...
  2020: 0 transfers
Fetching SRS and recruiting for 2021...
  2021: 619 transfers
Fetching SRS and recruiting for 2022...
  2022: 728 transfers
Fetching SRS and recruiting for 2023...
  2023: 933 transfers
Fetching SRS and recruiting for 2024...
  2024: 1211 transfers
Fetching SRS and recruiting for 2025...
  2025: 1611 transfers
Fetching SRS and recruiting for 2026...
  2026: 1424 transfers

SRS rows:        2147
Recruiting rows: 1139
WAB rows:        2512

Portal rows: 6526


In [13]:
transfers_in = portal_df.groupby(['Season', 'destination']).agg(
    transfers_in=('origin', 'count'),
    transfer_in_stars=('stars', 'mean')
).reset_index().rename(columns={'destination': 'Team'})

transfers_out = portal_df.groupby(['Season', 'origin']).agg(
    transfers_out=('destination', 'count')
).reset_index().rename(columns={'origin': 'Team'})
portal_agg = transfers_in.merge(transfers_out, on=['Season', 'Team'], how='outer').fillna(0)
portal_agg['net_transfers'] = portal_agg['transfers_in'] - portal_agg['transfers_out']

In [14]:
model_df = Wab_data.copy()
model_df = model_df.merge(srs_df,        on=['Season', 'Team'], how='left')
model_df = model_df.merge(recruiting_df, on=['Season', 'Team'], how='left')
model_df = model_df.merge(portal_agg,    on=['Season', 'Team'], how='left')

model_df['transfers_in']      = model_df['transfers_in'].fillna(0)
model_df['transfers_out']     = model_df['transfers_out'].fillna(0)
model_df['net_transfers']     = model_df['net_transfers'].fillna(0)
model_df['transfer_in_stars'] = model_df['transfer_in_stars'].fillna(0)

print("After merge:", model_df.shape)

After merge: (2512, 16)


In [17]:
model_df = model_df.sort_values(['Team', 'Season']).reset_index(drop=True)

model_df['wab_lag1']               = model_df.groupby('Team')['Wab'].shift(1)
model_df['wab_lag2']               = model_df.groupby('Team')['Wab'].shift(2)
model_df['srs_lag1']               = model_df.groupby('Team')['SRS'].shift(1)
model_df['srs_lag2']               = model_df.groupby('Team')['SRS'].shift(2)
model_df['elo_lag1']               = model_df.groupby('Team')['Elo'].shift(1)
model_df['recruiting_rating_lag1'] = model_df.groupby('Team')['Recruiting_Rating'].shift(1)
model_df['transfers_in_lag1']      = model_df.groupby('Team')['transfers_in'].shift(1)
model_df['transfers_out_lag1']     = model_df.groupby('Team')['transfers_out'].shift(1)
model_df['net_transfers_lag1']     = model_df.groupby('Team')['net_transfers'].shift(1)
model_df['transfer_stars_lag1']    = model_df.groupby('Team')['transfer_in_stars'].shift(1)
model_df['wab_next']               = model_df.groupby('Team')['Wab'].shift(-1)

model_df_clean = model_df.dropna(subset=['wab_next', 'wab_lag1', 'srs_lag1']).copy()
model_df_clean['wab_lag2']               = model_df_clean['wab_lag2'].fillna(model_df_clean['wab_lag1'])
model_df_clean['srs_lag2']               = model_df_clean['srs_lag2'].fillna(model_df_clean['srs_lag1'])
model_df_clean['recruiting_rating_lag1'] = model_df_clean['recruiting_rating_lag1'].fillna(
    model_df_clean['recruiting_rating_lag1'].median()
)

print(f"Clean model rows: {len(model_df_clean)}")

Clean model rows: 1779


In [18]:
features = [
    'wab_lag1',
    'wab_lag2',
    'srs_lag1',
    'srs_lag2',
    'recruiting_rating_lag1',
    'transfers_in_lag1',
    'transfers_out_lag1',
    'net_transfers_lag1',
    'transfer_stars_lag1',
]

X        = model_df_clean[features]
y        = model_df_clean['wab_next']
wab_mean = y.mean()
y_centered = y - wab_mean

scaler = StandardScaler()
gbr    = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
tscv   = TimeSeriesSplit(n_splits=4)

mae_scores = []
r2_scores  = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train_s = scaler.fit_transform(X.iloc[train_idx])
    X_test_s  = scaler.transform(X.iloc[test_idx])
    gbr.fit(X_train_s, y_centered.iloc[train_idx])
    preds = gbr.predict(X_test_s) + wab_mean
    mae_scores.append(mean_absolute_error(y.iloc[test_idx], preds))
    r2_scores.append(r2_score(y.iloc[test_idx], preds))
    print(f"Fold {fold+1} — MAE: {mae_scores[-1]:.3f}, R²: {r2_scores[-1]:.3f}")

print(f"\nAverage MAE: {sum(mae_scores)/len(mae_scores):.3f}")
print(f"Average R²:  {sum(r2_scores)/len(r2_scores):.3f}")

importance = pd.DataFrame({
    'feature':    features,
    'importance': gbr.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature importances:")
print(importance.to_string(index=False))

Fold 1 — MAE: 4.128, R²: 0.423
Fold 2 — MAE: 4.327, R²: 0.303
Fold 3 — MAE: 4.276, R²: 0.257
Fold 4 — MAE: 3.810, R²: 0.359

Average MAE: 4.136
Average R²:  0.336

Feature importances:
               feature  importance
              srs_lag1    0.307153
              wab_lag1    0.225996
              srs_lag2    0.151072
recruiting_rating_lag1    0.142666
              wab_lag2    0.090060
     transfers_in_lag1    0.036744
   transfer_stars_lag1    0.020480
    net_transfers_lag1    0.013807
    transfers_out_lag1    0.012022


In [21]:
elo_mean = model_df_clean['elo_lag1'].mean()
elo_std  = model_df_clean['elo_lag1'].std()
srs_mean = model_df_clean['srs_lag1'].mean()
srs_std  = model_df_clean['srs_lag1'].std()

pred_df = model_df[model_df['Season'] == 2026].copy()
pred_df['wab_lag2']               = pred_df['wab_lag2'].fillna(pred_df['wab_lag1'])
pred_df['recruiting_rating_lag1'] = pred_df['recruiting_rating_lag1'].fillna(
    model_df_clean['recruiting_rating_lag1'].median()
)

# Fill missing SRS with ELO normalized to SRS scale
pred_df['srs_lag1'] = pred_df['srs_lag1'].fillna(
    (pred_df['elo_lag1'] - elo_mean) / elo_std * srs_std + srs_mean
)
pred_df['srs_lag2'] = pred_df['srs_lag2'].fillna(pred_df['srs_lag1'])

pred_df = pred_df.dropna(subset=['wab_lag1', 'srs_lag1'])

X_pred                        = pred_df[features]
X_pred_scaled                 = scaler.transform(X_pred)
pred_df['predicted_wab_2027'] = (gbr.predict(X_pred_scaled) + wab_mean).round(3)

predictions_2027 = (
    pred_df[['Team', 'Wab', 'SRS', 'Recruiting_Rating', 'transfers_in',
             'transfers_out', 'net_transfers', 'predicted_wab_2027']]
    .rename(columns={
        'Wab':               'wab_2026',
        'SRS':               'srs_2026',
        'Recruiting_Rating': 'recruiting_2026',
        'transfers_in':      'transfers_in_2026',
        'transfers_out':     'transfers_out_2026',
        'net_transfers':     'net_transfers_2026',
    })
    .sort_values('predicted_wab_2027', ascending=False)
    .reset_index(drop=True)
)
predictions_2027.insert(0, 'rank', predictions_2027.index + 1)

print("\nTop 25 Predicted WAB for 2027 Season\n")
print(predictions_2027.head(60).to_string(index=False))


Top 25 Predicted WAB for 2027 Season

 rank              Team  wab_2026  srs_2026  recruiting_2026  transfers_in_2026  transfers_out_2026  net_transfers_2026  predicted_wab_2027
    1           Alabama     6.774       NaN            63.81                4.0                 3.0                 1.0               8.388
    2           Houston     7.668       NaN            52.23                4.0                 2.0                 2.0               8.195
    3            Auburn     0.604       NaN            20.10                5.0                 4.0                 1.0               6.941
    4        Iowa State     6.547       NaN            58.86                5.0                 0.0                 5.0               6.643
    5              Duke    10.900       NaN            69.16                2.0                 2.0                 0.0               6.377
    6          Maryland    -6.471       NaN            64.95                6.0                 1.0                 5.0  

In [22]:
actual_2026 = Wab_data[Wab_data['Season'] == 2026][['Team', 'Wab']].rename(
    columns={'Wab': 'actual_wab_2026'}
)
pred_2026_df = model_df[model_df['Season'] == 2025].copy()
pred_2026_df['wab_lag2']               = pred_2026_df['wab_lag2'].fillna(pred_2026_df['wab_lag1'])
pred_2026_df['srs_lag2']               = pred_2026_df['srs_lag2'].fillna(pred_2026_df['srs_lag1'])
pred_2026_df['recruiting_rating_lag1'] = pred_2026_df['recruiting_rating_lag1'].fillna(
    model_df_clean['recruiting_rating_lag1'].median()
)
pred_2026_df = pred_2026_df.dropna(subset=['wab_lag1', 'srs_lag1'])

X_2026        = pred_2026_df[features]
X_2026_scaled = scaler.transform(X_2026)
pred_2026_df['predicted_wab_2026'] = (gbr.predict(X_2026_scaled) + wab_mean).round(3)

comparison = pred_2026_df[['Team', 'predicted_wab_2026']].merge(actual_2026, on='Team', how='inner')
comparison['error']     = (comparison['predicted_wab_2026'] - comparison['actual_wab_2026']).round(3)
comparison['abs_error'] = comparison['error'].abs().round(3)

print(f"\nValidation vs actual 2026:")
print(f"MAE:         {comparison['abs_error'].mean():.3f}")
print(f"Correlation: {comparison['predicted_wab_2026'].corr(comparison['actual_wab_2026']):.3f}")
print(f"\nTeams within 2 WAB: {(comparison['abs_error'] <= 2).sum()}")
print(f"Teams within 4 WAB: {(comparison['abs_error'] <= 4).sum()}")
print(f"Teams within 6 WAB: {(comparison['abs_error'] <= 6).sum()}")
print(f"Teams over 6 WAB:   {(comparison['abs_error'] > 6).sum()}")


Validation vs actual 2026:
MAE:         3.374
Correlation: 0.780

Teams within 2 WAB: 127
Teams within 4 WAB: 235
Teams within 6 WAB: 312
Teams over 6 WAB:   50


In [23]:
predictions_2027.to_csv("data/predictions_2027.csv", index=False)